In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

def load_and_preprocess_data(filepath):
    """Loads dataset and applies all necessary encodings."""
    print("Loading and preprocessing data...")
    df = pd.read_csv(filepath)

    # 1. Drop identifiers and redundant text columns
    cols_to_drop = [
        'Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 
        'Locality', 'Locality Verbose', 'Cuisines', 'Currency', 
        'Rating color', 'Complexity_Label', 'Switch to order menu'
    ]
    df_dropped = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # 2. Handle Yes/No binary columns
    binary_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now']
    for col in binary_cols:
        if col in df_dropped.columns:
            df_dropped[col] = df_dropped[col].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}).fillna(0).astype(int)

    # 3. Label Encoding for Ordinal variables
    label_cols = ['Rating text']
    le = LabelEncoder()
    for col in label_cols:
        if col in df_dropped.columns:
            df_dropped[col] = le.fit_transform(df_dropped[col].astype(str))

    # 4. One-Hot Encoding for Nominal variables
    nominal_cols = ['Day_of_Week', 'Time_Slot', 'Weather_Condition']
    df_encoded = pd.get_dummies(df_dropped, columns=[c for c in nominal_cols if c in df_dropped.columns])

    # Convert all resulting boolean columns to integers
    for col in df_encoded.columns:
        if df_encoded[col].dtype == 'bool':
            df_encoded[col] = df_encoded[col].astype(int)

    # Ensure everything is strictly numeric
    df_encoded = df_encoded.apply(pd.to_numeric, errors='coerce').fillna(0)
    
    return df_encoded

def train_and_save_ridge_model(df):
    """Trains the Ridge model, applies standard scaling, evaluates, and saves artifacts."""
    print("Training Ridge Regression model...")
    
    # Define features (X) and target (y)
    X = df.drop(columns=['Actual_KPT_Minutes'])
    y = df['Actual_KPT_Minutes']
    feature_names = X.columns

    # Split into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # --- CRITICAL: Initialize and apply StandardScaler ---
    # Ridge Regression requires scaled data to apply L2 penalty correctly
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test) # Only transform the test set to prevent data leakage

    # Initialize and train Ridge model
    model = Ridge(alpha=1.0, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Make predictions
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n--- Model Evaluation (Ridge Regression) ---")
    print(f"R² Score (Accuracy):  {r2:.4f}")
    print(f"Mean Absolute Error:  {mae:.4f} minutes")
    print(f"Root Mean Sq. Error:  {rmse:.4f} minutes")
    
    # Save the model, the scaler, and feature columns for future deployment
    joblib.dump(model, 'zomato_kpt_ridge_model.pkl')
    joblib.dump(scaler, 'zomato_kpt_scaler.pkl')
    joblib.dump(list(feature_names), 'kpt_model_features.pkl')
    
    print("\n✅ Deployment Pipeline artifacts successfully saved!")
    print(" 1. Model:    'zomato_kpt_ridge_model.pkl'")
    print(" 2. Scaler:   'zomato_kpt_scaler.pkl'")
    print(" 3. Features: 'kpt_model_features.pkl'")

if __name__ == "__main__":
    # Define file path to your dataset
    dataset_path = 'zomato_KPT_Final_Dataset.csv'
    
    # Run the end-to-end pipeline
    processed_data = load_and_preprocess_data(dataset_path)
    train_and_save_ridge_model(processed_data)

Loading and preprocessing data...
Training Ridge Regression model...

--- Model Evaluation (Ridge Regression) ---
R² Score (Accuracy):  0.9758
Mean Absolute Error:  1.4855 minutes
Root Mean Sq. Error:  1.7166 minutes

✅ Deployment Pipeline artifacts successfully saved!
 1. Model:    'zomato_kpt_ridge_model.pkl'
 2. Scaler:   'zomato_kpt_scaler.pkl'
 3. Features: 'kpt_model_features.pkl'
